<a href="https://colab.research.google.com/github/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/05_kendi_fotograflarinla_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a> &nbsp; <a href="https://github.com/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/05_kendi_fotograflarinla_egitim.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-Kaynak-181717?logo=github" alt="GitHub'da Görüntüle"/></a>

> 🚀 **Bu notebook'u tarayıcıda çalıştırmak için sol üstteki "Colab'da Aç" butonuna tıklayın** — hiçbir şey kurmanıza gerek yok!


<div style="background: linear-gradient(135deg, #0B3D91 0%, #1565C0 60%, #1E88E5 100%); padding: 26px 30px; border-radius: 14px; color: white; margin: 0 0 24px 0; font-family: Georgia, serif; box-shadow: 0 4px 14px rgba(11,61,145,0.25);">

<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
  <span style="font-size: 11px; font-weight: bold; letter-spacing: 2px; color: #BBDEFB;">MEB · YEĞİTEK · DERİN ÖĞRENME SERİSİ</span>
  <span style="font-size: 11px; background: rgba(255,255,255,0.18); padding: 5px 12px; border-radius: 20px; color: white; font-weight: bold;">DERS 5 / 5</span>
</div>

<h1 style="margin: 8px 0 4px 0; color: white; font-size: 28px; line-height: 1.2;">🌺 Kendi Fotoğraflarınla Profesyonel Sınıflandırıcı</h1>

<p style="margin: 8px 0 0 0; color: #E3F2FD; font-size: 13px;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong> — Öğretmenler için Uygulamalı Yapay Zeka<br>
<span style="color: #BBDEFB;">⏱️ Süre: ~70 dakika · 🖥️ Ortam: Google Colab (GPU önerilir)</span>
</p>

<div style="border-top: 1px solid rgba(255,255,255,0.22); margin-top: 14px; padding-top: 12px; font-size: 12px; color: #E3F2FD;">
🎯 Bu notebook <strong>non-teknik kitle</strong> için hazırlanmıştır · Kavramlar günlük hayattan analojilerle anlatılır · Kod minimum, açıklama maksimumdur.
</div>

</div>

## 🎯 Bu Notebook'ta Ne Öğreneceğiz?

Bu serinin **son notebook'u**. Şimdiye kadar hep hazır veri setleri kullandık. Şimdi sıra **gerçek dünyaya** geliyor:

- 🌐 **Transfer Learning** — milyonlarca resimle eğitilmiş hazır modelleri kullanmak
- 🧠 **MobileNetV2** — Google'ın eğittiği güçlü ön-eğitimli model
- 🎨 **Data Augmentation** — az veriyle çok şey öğretmek (resmi döndür, çevir, kaydır)
- ❄️ **Frozen layers** — modelin bir kısmını "dondurmak"
- 🌺 **Çiçek tanıma** — `tf_flowers` veri setiyle (5 çiçek türü)
- 🌐 Gradio ile **profesyonel arayüz** — webcam veya dosya yüklemeli

Sonunda elinizde **gerçek ürün seviyesinde** bir AI uygulaması olacak.

## 🤔 Önce Bir Senaryo

Diyelim ki MEB için bir uygulama yazıyorsunuz: **"Öğrenciler bir fotoğraf çeksin, hangi çiçek olduğunu öğrensin."**

**Sıfırdan model eğitmek için ne lazım?**
- Yüzbinlerce çiçek fotoğrafı 📸
- Haftalarca eğitim süresi ⏳
- Pahalı GPU'lar 💸
- Görüntü işleme uzmanı 👨‍🔬

**Bu kaynaklara sahip misiniz?** Muhtemelen hayır.

---

### 💡 Çözüm: Transfer Learning

Google, Facebook gibi şirketler **milyonlarca** resimle modeller eğitti (ImageNet veri seti — 14 milyon resim, 1000 sınıf). Bu modeller "dünyayı görmeyi" öğrendi.

**Biz bu modellerin "görme bölümünü" alıyoruz, sadece son katmanı kendi probleme göre eğitiyoruz.**

> 🎓 **Analoji:** Bir tıp doktoru düşünün. 6 yıl tıp okudu (genel görüntü tanıma), şimdi cilt hastalıkları uzmanı olmak istiyor (özel sınıflandırma). 6 yılı tekrar etmesine gerek yok — sadece **uzmanlaşma kısmını** öğrenmesi yeter. Transfer learning aynen böyle çalışır.


## 📦 Kurulum

In [ ]:
!pip install -q gradio tensorflow-datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import gradio as gr

keras.utils.set_random_seed(42)
print(f"✅ TensorFlow {tf.__version__} hazır.")
print(f"✅ GPU mevcut mu? {'Evet 🚀' if tf.config.list_physical_devices('GPU') else 'Hayır (CPU ile çalışacak)'}")

## 📊 Veri Seti — TF Flowers

`tf_flowers` veri seti **5 çiçek türü** içerir:
1. 🌼 daisy (papatya)
2. 🌻 dandelion (karahindiba)
3. 🌹 roses (gül)
4. 🌻 sunflowers (ayçiçeği)
5. 🌷 tulips (lale)

Toplam ~3.670 fotoğraf. Hiç indirmek zorunda değiliz — TensorFlow Datasets bizim için halleder.

In [ ]:
# Veri setini indir (ilk seferde ~218 MB, sonra cache'ten okur)
veri, info = tfds.load('tf_flowers', with_info=True, as_supervised=True)

sinif_isimleri = info.features['label'].names
print(f"🌺 Sınıflar: {sinif_isimleri}")
print(f"📦 Toplam örnek: {info.splits['train'].num_examples}")

### Eğitim/Doğrulama Olarak Bölelim

In [ ]:
# %80 eğitim, %20 doğrulama
veri_train, veri_val = tfds.load(
    'tf_flowers',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True
)

# Birkaç örnek görelim
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, (resim, etiket) in enumerate(veri_train.take(10)):
    ax = axes[i//5, i%5]
    ax.imshow(resim.numpy())
    ax.set_title(sinif_isimleri[etiket.numpy()], fontsize=11)
    ax.axis('off')
plt.suptitle('🌺 TF Flowers Örnekleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Veri Hazırlığı (Pipeline)

MobileNetV2 **224×224** boyutunda resim bekler. Hazırlayalım:

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def hazirla(resim, etiket):
    resim = tf.image.resize(resim, (IMG_SIZE, IMG_SIZE))
    resim = tf.cast(resim, tf.float32)
    return resim, etiket

train_ds = (veri_train
            .map(hazirla, num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(1000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (veri_val
          .map(hazirla, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

print("✅ Pipeline hazır.")

## 🎨 Data Augmentation — Az Veriyle Çok Veri

3.670 fotoğraf yetersiz olabilir. Çözüm: **mevcut fotoğrafları çevir, döndür, parlaklığını değiştir** → sanki yeni veri varmış gibi davran.

> 🎓 **Analoji:** Öğrenciye 10 farklı dilbilgisi alıştırması yerine, aynı 10 cümlenin farklı dizilişlerini vermek. Aynı kuralı pekiştirir.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

# Bir örnek üzerinde augmentation'ı görselleştir
ornek_resim, _ = next(iter(train_ds.take(1)))
ornek_resim = ornek_resim[0:1]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(ornek_resim[0].numpy().astype('uint8'))
axes[0].set_title('Orijinal', fontweight='bold')
axes[0].axis('off')

for i in range(4):
    augmented = data_augmentation(ornek_resim, training=True)
    axes[i+1].imshow(augmented[0].numpy().astype('uint8'))
    axes[i+1].set_title(f'Augmented {i+1}')
    axes[i+1].axis('off')

plt.suptitle('🎨 Aynı Resimden Farklı Versiyonlar', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🧠 Transfer Learning Modeli

### Adım 1: Hazır Modeli Yükle

MobileNetV2'yi **ImageNet ağırlıklarıyla** yükleyeceğiz, ama **son katmanını çıkaracağız** (kendi sınıflarımızı koyacağız).

In [ ]:
# include_top=False → son sınıflandırma katmanı YOK
temel_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Bu katmanı dondur — eğitimde değişmesin
temel_model.trainable = False

print(f"✅ MobileNetV2 yüklendi.")
print(f"   Toplam {len(temel_model.layers)} katman var, hepsi DONDURULDU (eğitimde değişmeyecek).")

### Adım 2: Kendi Sınıflandırma Katmanımızı Ekle

**Mimari:**
```
[224x224x3 resim]
   ↓ Augmentation (sadece eğitimde aktif)
   ↓ Preprocess (MobileNetV2'nin beklediği formata çevir)
   ↓ MobileNetV2 (DONDURULMUŞ)
[7x7x1280 özellikler]
   ↓ GlobalAveragePooling2D
[1280]
   ↓ Dropout(0.2)
   ↓ Dense(5, Softmax)
[5 çiçek türü olasılığı]
```

In [ ]:
girdi = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(girdi)
x = keras.applications.mobilenet_v2.preprocess_input(x)  # -1 ile 1 arasına ölçekler
x = temel_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
cikti = layers.Dense(len(sinif_isimleri), activation='softmax')(x)

model = keras.Model(girdi, cikti)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Eğitilebilir parametre sayısına bakalım
model.summary()

### 🔥 Önemli Gözlem

`model.summary()` çıktısında dikkat: **Trainable params** (eğitilebilir) çok az, **Non-trainable params** (donmuş) milyonlarca.

Yani biz sadece **son katmanın ağırlıklarını** öğreniyoruz. Bu yüzden:
- ✅ Çok hızlı eğitilir
- ✅ Az veri yeterli olur
- ✅ MobileNetV2'nin "dünyayı tanıma" gücünden faydalanırız

## 🚀 Eğitim — Sadece Son Katman

In [ ]:
gecmis = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    verbose=1
)

In [ ]:
# Eğitim grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(gecmis.history['accuracy'], label='Eğitim', linewidth=2)
axes[0].plot(gecmis.history['val_accuracy'], label='Doğrulama', linewidth=2)
axes[0].set_title('Doğruluk', fontweight='bold'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[0].set_ylim([0, 1])

axes[1].plot(gecmis.history['loss'], label='Eğitim', linewidth=2)
axes[1].plot(gecmis.history['val_loss'], label='Doğrulama', linewidth=2)
axes[1].set_title('Hata', fontweight='bold'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

print(f"\n🎯 Doğrulama doğruluğu: {gecmis.history['val_accuracy'][-1] * 100:.1f}%")
print(f"   (Sıfırdan eğitsek aynı sonuca ulaşmak SAATLER sürerdi)")

## 🔍 Birkaç Tahmin Görelim

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
ornekler = list(val_ds.unbatch().take(8))

for i, (resim, gercek) in enumerate(ornekler):
    ax = axes[i//4, i%4]

    img_input = tf.expand_dims(resim, 0)
    olasiliklar = model.predict(img_input, verbose=0)[0]
    tahmin = np.argmax(olasiliklar)
    guven = olasiliklar[tahmin] * 100

    ax.imshow(resim.numpy().astype('uint8'))
    renk = 'green' if tahmin == gercek.numpy() else 'red'
    ax.set_title(f'Tahmin: {sinif_isimleri[tahmin]} (%{guven:.0f})\nGerçek: {sinif_isimleri[gercek.numpy()]}',
                 color=renk, fontsize=10)
    ax.axis('off')

plt.suptitle('🌺 Doğrulama Setinde Tahminler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 💾 Modeli Kaydet

In [ ]:
model.save('cicek_modeli.keras')
import os
boyut_mb = os.path.getsize('cicek_modeli.keras') / (1024 * 1024)
print(f"✅ Model kaydedildi: cicek_modeli.keras ({boyut_mb:.1f} MB)")

## 🌐 Profesyonel Gradio Arayüzü

Şimdi son rötuşlar:
- Resim yükleme **VE** webcam desteği
- En olası **3 sınıf** ve emoji ile renkli çıktı
- Kullanıcı dostu açıklama

In [ ]:
def cicek_tahmin_et(resim):
    if resim is None:
        return {sinif: 0.0 for sinif in sinif_isimleri}

    img = tf.image.resize(resim, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.expand_dims(img, 0)

    olasiliklar = model.predict(img, verbose=0)[0]

    emoji = {'daisy': '🌼', 'dandelion': '🌻', 'roses': '🌹',
             'sunflowers': '🌻', 'tulips': '🌷'}
    tr_isim = {'daisy': 'Papatya', 'dandelion': 'Karahindiba',
               'roses': 'Gül', 'sunflowers': 'Ayçiçeği', 'tulips': 'Lale'}

    sonuc = {}
    for i, sinif in enumerate(sinif_isimleri):
        etiket = f"{emoji.get(sinif, '🌸')} {tr_isim.get(sinif, sinif)}"
        sonuc[etiket] = float(olasiliklar[i])
    return sonuc


arayuz = gr.Interface(
    fn=cicek_tahmin_et,
    inputs=gr.Image(type='numpy', label='🌺 Çiçek fotoğrafı yükleyin veya webcam kullanın',
                    sources=['upload', 'webcam'], height=300),
    outputs=gr.Label(num_top_classes=3, label='🎯 En Olası 3 Çiçek'),
    title='🌺 MEB YEĞİTEK · Akıllı Çiçek Tanıyıcı',
    description=('**Transfer Learning + MobileNetV2** ile eğitilmiş model. '
                 'Bahçeden / parktan bir çiçek fotoğrafı yükleyin, model size türünü söyleyecek. '
                 'Tanınan türler: papatya, karahindiba, gül, ayçiçeği, lale.'),
    examples=None,
    flagging_mode='never',
    theme=gr.themes.Soft()
)

arayuz.launch(share=False)

## 🎁 Bonus: Kendi 2 Sınıfınızı Eğitin

Bu pipeline'ı **kendi resimlerinizle** kullanmak çok kolay. İhtiyacınız:

1. İki klasör: `kedi/` ve `kopek/` (veya ne istiyorsanız)
2. Her klasörde 50-100 fotoğraf yeterli (transfer learning sayesinde!)
3. Kodun veri yükleme kısmını şu şekilde değiştirin:

```python
veri = keras.utils.image_dataset_from_directory(
    'kendi_klasorum/',
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='both',
    seed=42
)
```

Geri kalan her şey aynı! Bu, MEB YEĞİTEK kapsamında hazırlayabileceğiniz **eğitim materyali ya da öğrenci projeleri** için harika bir başlangıçtır.

## 📝 Final Özet — 5 Notebook'ta Ne Öğrendik?

| Notebook | Konu | Anahtar Kavram |
|---|---|---|
| **01** | Tek Nöron | Ağırlık, bias, gradyan iniş |
| **02** | Çok Katmanlı Ağ | Katmanlar, aktivasyon, sınıflandırma |
| **03** | Gradio + Kayıt | Modeli üretime alma, web arayüzü |
| **04** | CNN | Konvolüsyon, görüntü için özel mimari |
| **05** | Transfer Learning | Hazır modelle az veriyle eğitim |

### 🎓 Final İçgörü

> **Derin öğrenme = Veri + Model + Bol Deney.**
>
> Sıfırdan başlamayın — **omuzlarda dur**. Hazır modeller, açık veri setleri, hazır kütüphaneler... hepsi sizin için var.

### 🚀 Sıradaki Adımlar

- 📚 `kaynaklar.md` dosyasındaki ileri seviye kaynakları inceleyin
- 🌐 [Hugging Face Spaces](https://huggingface.co/spaces) ile modelinizi internete bedava yayınlayın
- 👨‍🏫 Öğrencilerinizle bir AI projesi yapın
- 💬 [TensorFlow Türkçe topluluğu](https://www.tensorflow.org/community)nda sorular sorun

**Tebrikler — artık derin öğrenmeyi sadece anlamıyor, uygulayabiliyorsunuz! 🎉**

<div style="background: #0B3D91; color: #BBDEFB; padding: 26px 30px; border-radius: 14px; margin-top: 32px; font-family: Georgia, serif;">

<h3 style="color: white; margin: 0 0 12px 0; border-bottom: 2px solid #FFC107; padding-bottom: 8px; display: inline-block;">MEB · YEĞİTEK</h3>

<p style="font-family: Calibri, sans-serif; font-size: 14px; color: #E3F2FD; margin: 0 0 8px 0; line-height: 1.6;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong><br>
Öğretmen ve eğitimciler için uygulamalı derin öğrenme eğitim serisi.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 12px; color: #BBDEFB; margin: 12px 0 0 0;">
🎓 Bu notebook eğitim amaçlıdır · Kullanılan tüm kütüphaneler açık kaynaktır.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 11px; color: #90CAF9; margin: 14px 0 0 0; text-align: center; border-top: 1px solid rgba(255,255,255,0.15); padding-top: 12px;">
© 2026 MEB YEĞİTEK · Derin Öğrenme Serisi
</p>

</div>